# Add elevation

In [3]:
import pandas as pd
import numpy as np
import requests
import json

In [2]:
df = pd.read_excel('etv_sura.xlsx')
df.head()

,año,semana,temp,año_semana,prcp,count,codigo,MPIO_CCNCT,MPIO_CNMBR,geometry,longitude,latitude
0,2014,1,25.484035,2014-01,3.996071,10,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848
1,2014,2,25.170785,2014-02,4.486607,5,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848
2,2014,3,25.936244,2014-03,0.098929,7,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848
3,2014,4,25.173205,2014-04,3.746250,8,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848
4,2014,5,25.489607,2014-05,0.231071,9,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848


In [4]:
# 2. Prepare the data for the API request
# The API expects a JSON object with a 'locations' key
locations_payload = {
    "locations": df[['latitude', 'longitude']].to_dict(orient='records')
}

In [5]:
# 3. Make the POST request to the Open-Elevation API
url = "https://api.open-elevation.com/api/v1/lookup"
headers = {'Content-type': 'application/json'}

try:
    response = requests.post(url, headers=headers, data=json.dumps(locations_payload))
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

    # 4. Process the response and add it to the DataFrame
    results = response.json()['results']
    
    # Extract the elevation from each result dictionary
    elevations = [r['elevation'] for r in results]
    
    # Add the list of elevations as a new column
    df['elevation_m'] = elevations
    
    print("Successfully added elevation data:")
    print(df)

except requests.exceptions.RequestException as e:
    print(f"An error occurred with the API request: {e}")
except (KeyError, IndexError) as e:
    print(f"Error processing the API response: {e}")

Successfully added elevation data:
        año  semana       temp año_semana      prcp  count  codigo  \
0      2014       1  25.484035    2014-01  3.996071     10    5045   
1      2014       2  25.170785    2014-02  4.486607      5    5045   
2      2014       3  25.936244    2014-03  0.098929      7    5045   
3      2014       4  25.173205    2014-04  3.746250      8    5045   
4      2014       5  25.489607    2014-05  0.231071      9    5045   
...     ...     ...        ...        ...       ...    ...     ...   
12523  2023      48  27.113556    2023-48  5.157500     25    5837   
12524  2023      49  27.449838    2023-49  1.753393     29    5837   
12525  2023      50  27.164132    2023-50  4.684821     22    5837   
12526  2023      51  26.912938    2023-51  3.640893     30    5837   
12527  2023      52  26.505585    2023-52  7.477500     36    5837   

       MPIO_CCNCT MPIO_CNMBR  \
0            5045   APARTADÓ   
1            5045   APARTADÓ   
2            5045   APARTADÓ

In [9]:
# --- Step 1: Create a date column from 'año' and 'semana' ---

# We'll create a date string and specify the format. 
# '%Y' is for the year, '%W' for the week number (Monday as the first day).
# We add '-1' to signify Monday (day of the week, 0=Sunday...6=Saturday, but with %W, 1=Monday).
date_str = df['año'].astype(str) + '-' + df['semana'].astype(str) + '-1'
df['date'] = pd.to_datetime(date_str, format='%Y-%W-%w')

print("DataFrame after adding the 'date' column:")
df.head()


DataFrame after adding the 'date' column:


,año,semana,temp,año_semana,prcp,count,codigo,MPIO_CCNCT,MPIO_CNMBR,geometry,longitude,latitude,elevation_m,date
0,2014,1,25.484035,2014-01,3.996071,10,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-06
1,2014,2,25.170785,2014-02,4.486607,5,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-13
2,2014,3,25.936244,2014-03,0.098929,7,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-20
3,2014,4,25.173205,2014-04,3.746250,8,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-27
4,2014,5,25.489607,2014-05,0.231071,9,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-02-03


In [10]:
# --- Step 2: Create shifted columns for 'count' (t+1 and t+2) ---

# It's crucial to sort the data by city and date before shifting
# to ensure the shifts are chronological within each group.
df.sort_values(by=['MPIO_CNMBR', 'date'], inplace=True)

# Group by city and then shift the 'count' column.
# shift(-1) looks at the *next* row in the group.
# shift(-2) looks two rows ahead in the group.
df['count_t+1'] = df.groupby('MPIO_CNMBR')['count'].shift(-1)
df['count_t+2'] = df.groupby('MPIO_CNMBR')['count'].shift(-2)


print("Final DataFrame with shifted columns:")
df.head()

Final DataFrame with shifted columns:


,año,semana,temp,año_semana,prcp,count,codigo,MPIO_CCNCT,MPIO_CNMBR,geometry,longitude,latitude,elevation_m,date,count_t+1,count_t+2
0,2014,1,25.484035,2014-01,3.996071,10,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-06,5.0,7.0
1,2014,2,25.170785,2014-02,4.486607,5,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-13,7.0,8.0
2,2014,3,25.936244,2014-03,0.098929,7,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-20,8.0,9.0
3,2014,4,25.173205,2014-04,3.746250,8,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-01-27,9.0,8.0
4,2014,5,25.489607,2014-05,0.231071,9,5045,5045,APARTADÓ,POLYGON ((-76.42525179693415 8.073453898570664...,-76.557904,7.895848,194.0,2014-02-03,8.0,3.0


In [11]:
df.describe()  # Display basic statistics of the DataFrame

,año,semana,temp,prcp,count,codigo,MPIO_CCNCT,longitude,latitude,elevation_m,date,count_t+1,count_t+2
count,12528.000000,12528.000000,12528.000000,12528.000000,12528.000000,12528.000000,12528.000000,12528.000000,12528.000000,12528.000000,12528,12504.000000,12480.000000
mean,2018.496169,26.601533,20.576758,6.517010,20.867736,28667.125000,28667.125000,-75.309712,6.712326,1150.000000,2018-12-30 07:35:10.344827648,20.855006,20.843349
min,2014.000000,1.000000,11.492567,0.000000,0.000000,5001.000000,5001.000000,-76.718654,3.399272,4.000000,2014-01-06 00:00:00,0.000000,0.000000
25%,2016.000000,14.000000,16.372103,2.413929,0.000000,5336.500000,5336.500000,-76.310919,5.723312,60.500000,2016-06-27 00:00:00,0.000000,0.000000
50%,2018.500000,27.000000,20.085619,5.605268,3.000000,8595.500000,8595.500000,-75.599568,6.310065,1287.500000,2018-12-31 00:00:00,3.000000,3.000000
75%,2021.000000,40.000000,25.581555,9.650134,14.000000,66501.000000,66501.000000,-74.803037,7.669112,1665.000000,2021-06-28 00:00:00,14.000000,14.000000
max,2023.000000,53.000000,29.437701,31.013034,1109.000000,76520.000000,76520.000000,-73.008474,10.985030,3369.000000,2023-12-25 00:00:00,1109.000000,1109.000000
std,2.871726,15.069353,4.612870,5.115679,59.554572,29240.132904,29240.132904,1.183609,2.026537,907.921921,NaN,59.576202,59.604604


In [12]:
df.dropna(subset=['elevation_m', 'count_t+1', 'count_t+2'], inplace=True)

In [13]:
df.to_csv('../platinum/data_sura.csv', index=False)